### Optuna
    - 최적의 파라미터를 찾기 위한 라이브러리
    - 별도의 라이브러리 설치
    - GridSearchCV와의 가장 큰 차이는 속도 Optuna 우세
    - GridSearch는 파라미터의 모든 조합을 fitting 확인
    - Optuna는 확률 기반 -> 모든 조합을 다 실행하지는 않는다
    - 조합 수 제어 -> GridSearch는 params로 제어, Optuna n_trials 매개변수로 제어

In [ ]:
# !pip install optuna


   ---------------------------------------- 0/5 [PyYAML]
   -------- ------------------------------- 1/5 [Mako]
   -------- ------------------------------- 1/5 [Mako]
   -------- ------------------------------- 1/5 [Mako]
   -------- ------------------------------- 1/5 [Mako]
   ---------------- ----------------------- 2/5 [colorlog]
   ------------------------ --------------- 3/5 [alembic]
   ------------------------ --------------- 3/5 [alembic]
   ------------------------ --------------- 3/5 [alembic]
   ------------------------ --------------- 3/5 [alembic]
   ------------------------ --------------- 3/5 [alembic]
   ------------------------ --------------- 3/5 [alembic]
   ------------------------ --------------- 3/5 [alembic]
   -------------------------------- ------- 4/5 [optuna]
   -------------------------------- ------- 4/5 [optuna]
   -------------------------------- ------- 4/5 [optuna]
   -------------------------------- ------- 4/5 [optuna]
   --------------------------


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import optuna
from sklearn.datasets import load_iris
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, make_scorer, f1_score

In [5]:
# iris 데이터 로드
X, Y = load_iris(return_X_y=True)

In [13]:
# objective 함수를 생성
# 매개변수는 필수
def objective(trial):
    # SVC 모델의 파라미터 경우의 수를 지정
    C = trial.suggest_float("C", 1e-3, 10.0, log=True)
    gamma = trial.suggest_float("gamma", 1e-4, 1.0, log=True)
    kernel = trial.suggest_categorical("kernel", ['linear', 'rbf'])
    model = SVC(C=C, gamma=gamma, kernel=kernel)

    pipe = Pipeline(
        [
            ('scaler', StandardScaler()),
            ('clf', model)
        ]
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = cross_val_score(pipe, X, Y, cv=cv, scoring=make_scorer(f1_score, average='macro'))
    return scores.mean()

In [14]:
# optuna을 이용해서 파라미터 찾기
study = optuna.create_study(direction='maximize', study_name='class_ml_tuning')
study.optimize(objective, n_trials=30, show_progress_bar=True)

[I 2025-10-31 17:40:42,141] A new study created in memory with name: class_ml_tuning
Best trial: 3. Best value: 0.959933:  20%|██        | 6/30 [00:00<00:00, 28.72it/s]

[I 2025-10-31 17:40:42,184] Trial 0 finished with value: 0.899635886067438 and parameters: {'C': 0.004147029407016015, 'gamma': 0.5743405011459997, 'kernel': 'rbf'}. Best is trial 0 with value: 0.899635886067438.
[I 2025-10-31 17:40:42,237] Trial 1 finished with value: 0.9187055447655798 and parameters: {'C': 0.0782243813828192, 'gamma': 0.5436073021396033, 'kernel': 'rbf'}. Best is trial 1 with value: 0.9187055447655798.
[I 2025-10-31 17:40:42,272] Trial 2 finished with value: 0.9392368647988516 and parameters: {'C': 0.039249550484934136, 'gamma': 0.000344313917182734, 'kernel': 'linear'}. Best is trial 2 with value: 0.9392368647988516.
[I 2025-10-31 17:40:42,301] Trial 3 finished with value: 0.9599331662489557 and parameters: {'C': 0.3948274675514613, 'gamma': 0.5622733806973343, 'kernel': 'linear'}. Best is trial 3 with value: 0.9599331662489557.
[I 2025-10-31 17:40:42,327] Trial 4 finished with value: 0.95328320802005 and parameters: {'C': 1.6022227817590518, 'gamma': 0.06451286709

Best trial: 10. Best value: 0.973266:  40%|████      | 12/30 [00:00<00:00, 28.72it/s]

[I 2025-10-31 17:40:42,396] Trial 6 finished with value: 0.858005115089514 and parameters: {'C': 0.0177744391408443, 'gamma': 0.06770612254493012, 'kernel': 'rbf'}. Best is trial 3 with value: 0.9599331662489557.
[I 2025-10-31 17:40:42,429] Trial 7 finished with value: 0.9599331662489557 and parameters: {'C': 0.3843049841899109, 'gamma': 0.6032414591344047, 'kernel': 'linear'}. Best is trial 3 with value: 0.9599331662489557.
[I 2025-10-31 17:40:42,463] Trial 8 finished with value: 0.9661728917348785 and parameters: {'C': 3.186124547690337, 'gamma': 0.0002678400915944943, 'kernel': 'linear'}. Best is trial 8 with value: 0.9661728917348785.
[I 2025-10-31 17:40:42,500] Trial 9 finished with value: 0.8651321398124466 and parameters: {'C': 0.6354094425956093, 'gamma': 0.0019360300699135691, 'kernel': 'rbf'}. Best is trial 8 with value: 0.9661728917348785.
[I 2025-10-31 17:40:42,534] Trial 10 finished with value: 0.9732664995822891 and parameters: {'C': 9.795551052058439, 'gamma': 0.00020366

Best trial: 12. Best value: 0.979849:  60%|██████    | 18/30 [00:00<00:00, 29.42it/s]

[I 2025-10-31 17:40:42,600] Trial 12 finished with value: 0.9798486114275586 and parameters: {'C': 8.670167424708815, 'gamma': 0.0013105332503183789, 'kernel': 'linear'}. Best is trial 12 with value: 0.9798486114275586.
[I 2025-10-31 17:40:42,633] Trial 13 finished with value: 0.9732664995822891 and parameters: {'C': 9.375921136441004, 'gamma': 0.0026696789303987116, 'kernel': 'linear'}. Best is trial 12 with value: 0.9798486114275586.
[I 2025-10-31 17:40:42,666] Trial 14 finished with value: 0.9661728917348785 and parameters: {'C': 1.950268980405919, 'gamma': 0.0014679698152743029, 'kernel': 'linear'}. Best is trial 12 with value: 0.9798486114275586.
[I 2025-10-31 17:40:42,699] Trial 15 finished with value: 0.8651321398124466 and parameters: {'C': 0.0010557537798433845, 'gamma': 0.009389374227686265, 'kernel': 'linear'}. Best is trial 12 with value: 0.9798486114275586.
[I 2025-10-31 17:40:42,730] Trial 16 finished with value: 0.9732664995822891 and parameters: {'C': 9.944931716895871,

Best trial: 12. Best value: 0.979849:  87%|████████▋ | 26/30 [00:00<00:00, 30.77it/s]

[I 2025-10-31 17:40:42,840] Trial 19 finished with value: 0.9661728917348785 and parameters: {'C': 3.97213789378588, 'gamma': 0.0006646734773377369, 'kernel': 'linear'}. Best is trial 12 with value: 0.9798486114275586.
[I 2025-10-31 17:40:42,872] Trial 20 finished with value: 0.9661728917348785 and parameters: {'C': 1.796670418690777, 'gamma': 0.039866295675979574, 'kernel': 'linear'}. Best is trial 12 with value: 0.9798486114275586.
[I 2025-10-31 17:40:42,903] Trial 21 finished with value: 0.9661728917348785 and parameters: {'C': 5.618942682808904, 'gamma': 0.004145836757406973, 'kernel': 'linear'}. Best is trial 12 with value: 0.9798486114275586.
[I 2025-10-31 17:40:42,933] Trial 22 finished with value: 0.9732664995822891 and parameters: {'C': 9.078834496297356, 'gamma': 0.002286106159939314, 'kernel': 'linear'}. Best is trial 12 with value: 0.9798486114275586.
[I 2025-10-31 17:40:42,965] Trial 23 finished with value: 0.9661728917348785 and parameters: {'C': 3.4794531551291525, 'gamm

Best trial: 12. Best value: 0.979849: 100%|██████████| 30/30 [00:01<00:00, 29.46it/s]

[I 2025-10-31 17:40:43,057] Trial 26 finished with value: 0.9254223367455297 and parameters: {'C': 3.4522179441354477, 'gamma': 0.004985044514651937, 'kernel': 'rbf'}. Best is trial 12 with value: 0.9798486114275586.
[I 2025-10-31 17:40:43,090] Trial 27 finished with value: 0.9599331662489557 and parameters: {'C': 0.12932710777535608, 'gamma': 0.0012201414254313025, 'kernel': 'linear'}. Best is trial 12 with value: 0.9798486114275586.
[I 2025-10-31 17:40:43,122] Trial 28 finished with value: 0.9661728917348785 and parameters: {'C': 2.167090723784209, 'gamma': 0.0031189853482894935, 'kernel': 'linear'}. Best is trial 12 with value: 0.9798486114275586.
[I 2025-10-31 17:40:43,158] Trial 29 finished with value: 0.8789225589225589 and parameters: {'C': 0.30312098462608206, 'gamma': 0.018626090319259967, 'kernel': 'rbf'}. Best is trial 12 with value: 0.9798486114275586.


In [15]:
# 최적의 파라미터 출력
print(study.best_params)

{'C': 8.670167424708815, 'gamma': 0.0013105332503183789, 'kernel': 'linear'}


In [17]:
# 최적의 스코어 확인
print(study.best_value)

0.9798486114275586
